# MNIST Unity Local PC Setup

This notebook documents how the local Windows Unity project was assembled for MNIST inference before performance measurements were collected.

The goal of this notebook is to reproduce the **working Unity application**, including:

- Unity project creation;
- required packages;
- ONNX model import;
- MNIST test-data import;
- scene and Canvas construction;
- laptop-camera access;
- 28×28 preprocessing;
- local MNIST inference;
- display of the live, processed, and frozen images; and
- generation of a Windows standalone build.

Performance measurements, CPU/GPU configuration changes, and benchmark procedures are documented separately in the measurement notebook.

## 1. Software Environment

The local PC implementation used:

- **Unity Editor 6000.3.22f1**
- **Unity Inference Engine / Sentis**
- **TextMeshPro**
- Windows standalone target
- a laptop-integrated webcam
- the MNIST ONNX model produced in the separate MNIST model workflow

The project was developed as a standard Unity desktop project and later extended for other deployment targets. This notebook documents only the Windows PC setup.

## 2. Create the Unity Project

Create a new Unity project using Unity Editor **6000.3.22f1**.

A standard 3D project is sufficient because the MNIST application primarily uses Unity UI elements and C# scripts.

After the project opens:

1. save the initial scene;
2. create an `Assets/Scripts` folder;
3. create an `Assets/Models` folder;
4. create an `Assets/TestData` folder; and
5. create an `Assets/Scenes` folder if the project does not already contain one.

For the archived reproducibility branch, the PC scene is stored as:

```text
FirstScene.unity
```

## 3. Install the Required Packages

Open:

```text
Window → Package Manager
```

### Unity Inference Engine

Install the Unity package that provides:

```csharp
using Unity.InferenceEngine;
```

The scripts use this package to:

- load the ONNX model;
- create an inference `Worker`;
- construct input tensors;
- schedule inference; and
- read the output tensor.

The source files therefore require the Unity Inference Engine package to compile.

### TextMeshPro

The user-interface scripts use TextMeshPro components through:

```csharp
using TMPro;
```

If TextMeshPro resources have not already been imported, Unity may prompt to import the required TMP essentials. Import the essentials before assigning TMP text components in the scene.

The benchmark and camera scripts can use `TMP_Text` fields to display predictions and status information.

## 4. Import the MNIST ONNX Model

Copy the MNIST model into the Unity project, for example:

```text
Assets/Models/mnist_model.onnx
```

Unity imports the ONNX file as a model asset that can be assigned to script fields of type:

```csharp
ModelAsset
```

The local implementation expects an input tensor with shape:

```text
1 × 784
```

and an output containing ten values corresponding to digits 0 through 9.

The model-generation and independent accuracy-validation workflow is documented in the separate MNIST model branches.

## 5. Import the MNIST Test Binaries

Several validation and benchmark scripts use the exported MNIST test set.

The repository stores the files together as:

```text
mnist_test_binaries.zip
```

Extract the archive and add:

```text
mnist_test_images.bin
mnist_test_labels.bin
```

to the Unity project, for example:

```text
Assets/TestData/
```

Unity imports these files as `TextAsset` resources.

The image binary contains 10,000 MNIST test images with 784 32-bit floating-point values per image. The label binary contains one label byte for each image.

These files are required for dataset-based validation scripts but are not required for the live webcam pipeline itself.

## 6. Add the PC Source Scripts

Add the following scripts to the project:

```text
CameraMNISTInference.cs
CameraPreprocessTest.cs
LaptopCameraTest.cs
MNISTAccuracyTest.cs
MNISTLatencyTest.cs
MNISTMemoryTest.cs
MNISTModelLoader.cs
MNISTSingleTest.cs
MNISTSustainedInference.cs
MNISTThroughputTest.cs
```

The first three scripts establish the camera pipeline incrementally:

```text
LaptopCameraTest
        |
        v
CameraPreprocessTest
        |
        v
CameraMNISTInference
```

The remaining scripts support model validation and later benchmark procedures.

This notebook focuses on getting the camera and inference application working. Benchmark activation and backend changes are covered in the separate measurement notebook.

## 7. Build the Scene Structure

The PC scene can be organized with the following top-level objects:

```text
Main Camera
Directional Light
Global Volume
MNIST Benchmark
Canvas
EventSystem
CameraManager
```

The archived scene is named:

```text
FirstScene
```

The Canvas was created directly in the scene rather than saved as a prefab.

Quest-specific objects are not part of this PC setup.

## 8. Create the Canvas

Create a Unity `Canvas` and add UI elements for the three stages of the camera pipeline.

A practical layout contains:

1. **Camera Display**  
   Shows the live webcam image.

2. **Processed Display**  
   Shows the current 28×28 preprocessed image.

3. **Frozen Display**  
   Shows the exact 28×28 image used when the user requests a classification.

4. **Prediction Text**  
   Displays the predicted digit and, when enabled, additional status information.

5. **Classify Button**  
   Freezes the current processed image and classifies that image.

Additional benchmark buttons may also be present in the final scene, but they are discussed in the measurement notebook.

The Canvas does not need to be converted to a prefab for reproduction. Its hierarchy and serialized component references are stored in `FirstScene.unity`.

## 9. Verify Basic Webcam Access

Before adding preprocessing or inference, test the PC camera using:

```text
LaptopCameraTest.cs
```

Attach the script to a scene object such as `CameraManager`.

Assign the live-camera `RawImage` to:

```text
cameraDisplay
```

The script:

- detects available webcams;
- selects the first device;
- requests a 640×480 stream;
- requests 30 FPS; and
- assigns the `WebCamTexture` to the UI.

The relevant camera request is:

```csharp
webcam = new WebCamTexture(
    devices[0].name,
    640,
    480,
    30
);
```

When this stage works, the live laptop-camera image should appear in the Canvas.

## 10. Verify the 28×28 Preprocessing Display

Next, test the image-processing stage independently using:

```text
CameraPreprocessTest.cs
```

This stage confirms that:

- camera pixels can be read from `WebCamTexture`;
- the image can be sampled into a 28×28 texture;
- RGB input can be converted to grayscale; and
- the processed image can be displayed through a second `RawImage`.

The target size is:

```csharp
private const int TargetSize = 28;
```

This early preprocessing test was useful before integrating the neural network because it separated camera/display problems from model-inference problems.

## 11. Configure the Final Camera Inference Pipeline

The combined local camera implementation is:

```text
CameraMNISTInference.cs
```

Attach this script to `CameraManager` or the scene object used to manage camera inference.

Assign the following references in the Inspector:

- `cameraDisplay`
- `processedDisplay`
- `predictionText`
- `modelAsset`
- `frozenDisplay`
- `classifyButton`

If the later benchmark controls are present in the scene, they may also be assigned, but they are not required simply to validate camera classification.

## 12. Camera Preprocessing Used for the Final Comparison

The final camera script was adjusted so that the PC image-processing path corresponded to the FPGA camera experiment.

The script treats the incoming webcam frame as a scaled representation of a 160×120 source coordinate space.

The FPGA-equivalent region is:

```text
x = 24 through 135
y = 4 through 115
```

Sampling occurs every four source-coordinate units in each direction.

This produces:

```text
28 × 28 = 784
```

input values.

The script computes grayscale using:

```text
0.299R + 0.587G + 0.114B
```

and then applies a binary threshold:

```text
gray < threshold → 0
gray >= threshold → 1
```

The default threshold field can be adjusted in the Unity Inspector.

## 13. Camera-to-Inference Data Flow

The completed PC pipeline is:

```text
Laptop Webcam
     |
     v
WebCamTexture
     |
     v
Read Camera Pixels
     |
     v
FPGA-Matched ROI Mapping
     |
     v
28 × 28 Sampling
     |
     v
Grayscale Conversion
     |
     v
Binary Threshold
     |
     v
784 Float Values
     |
     v
TensorShape(1, 784)
     |
     v
Unity Inference Engine
     |
     v
10 Output Values
     |
     v
Argmax
     |
     v
Predicted Digit
```

## 14. Freeze and Classify the Current Camera Frame

The `Classify` button calls the camera script's classification method.

The current 784-value preprocessed image is copied into a frozen buffer so the application can display exactly what was classified.

The input tensor is created as:

```csharp
using Tensor<float> inputTensor =
    new Tensor<float>(
        new TensorShape(1, 784),
        frozenInput
    );
```

Inference is then scheduled, the output is read back, and the largest of the ten output values determines the predicted digit.

The frozen-display panel is important because it provides a visual record of the precise binary input presented to the network.

## 15. Functional Validation

Before collecting performance measurements, verify the pipeline in stages:

### Stage 1 — Camera

Confirm that the live webcam image appears.

### Stage 2 — Processing

Confirm that the processed `RawImage` updates and displays a 28×28 representation of the camera frame.

### Stage 3 — Frozen Input

Press `Classify` and confirm that the frozen display shows the same 28×28 input used for the prediction.

### Stage 4 — Prediction

Present a clearly written digit and confirm that a prediction appears.

The purpose of this stage is functional validation, not benchmark collection.

## 16. Example Live-Inference Setup

A photographed example of the local PC application is included in the data branch.

![Unity live inference](Unity_Live_inference.jpg)

The display shows:

- the live camera image;
- the processed 28×28 image;
- the frozen classification image; and
- the predicted digit.

A wider photograph documents the physical workstation setup:

![Unity live inference overview](Unity_Live_Inference_Overview.jpg)

## 17. Windows Build Configuration

Once the application works correctly in the Unity Editor, create a Windows standalone build.

Open the Unity build interface and configure the project for:

```text
Windows
x86_64
```

Ensure that `FirstScene` is included in the build.

The Windows application should use the same:

- scene;
- ONNX model;
- scripts;
- UI references; and
- camera pipeline

that were validated in the Editor.

## 18. Build the Windows Application

Create a dedicated build directory outside the source folders.

A normal Windows Unity build produces an executable and supporting runtime files, typically including:

```text
MNISTUnity.exe
MNISTUnity_Data/
UnityPlayer.dll
```

and possibly additional Unity-generated runtime files.

The executable **cannot be distributed by itself** because the generated data directory and runtime dependencies are required.

For this repository, the complete Windows build is not included because the generated build package exceeds the repository upload-size limit used for this archive.

## 19. Running the Standalone Build

To reproduce the standalone application:

1. open the project in Unity Editor 6000.3.22f1;
2. confirm that the required packages are installed;
3. confirm that the ONNX model and required data assets are assigned;
4. open `FirstScene`;
5. confirm that the camera/UI references are assigned;
6. create a Windows x86-64 build;
7. keep all Unity-generated build files together; and
8. launch the generated executable.

Windows may request permission for the application to access the camera. Camera access must be allowed for the live inference pipeline to function.

## 20. Separation from the Measurement Notebook

This setup notebook ends once the PC application can:

- load the MNIST model;
- access the laptop camera;
- create the 28×28 binary image;
- display the processed and frozen images;
- classify the frozen image; and
- run as a Windows standalone application.

The second notebook documents:

- which benchmark scripts were active;
- CPU versus GPU Compute worker selection;
- Unity Editor changes between configurations;
- accuracy validation;
- latency;
- throughput;
- memory;
- sustained inference;
- camera-specific benchmarks;
- power measurement; and
- interpretation of the collected CSV files, screenshots, photographs, and videos.